# 05 — Task 2.3: Classical Machine Learning
Trains **Logistic Regression**, **Naive Bayes**, and **Linear SVM** with multiple feature representations (BoW, TF-IDF) and preprocessing configurations.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

from src.utils import load_data, evaluate_predictions, save_results, preprocess_corpus

import warnings
warnings.filterwarnings('ignore')

train_texts, train_labels = load_data('train')
test_texts,  test_labels  = load_data('test')
print(f'Train: {len(train_texts):,} | Test: {len(test_texts):,}')

## 1. Preprocessing variants
We build several preprocessed corpora to compare the effect of each technique.

In [ ]:
print('Preprocessing: baseline (lowercase only)...')
train_base = preprocess_corpus(train_texts, lowercase=True)
test_base  = preprocess_corpus(test_texts,  lowercase=True)

print('Preprocessing: +stopwords +lemma...')
train_full = preprocess_corpus(train_texts, lowercase=True,
                                remove_stopwords=True, lemmatize=True)
test_full  = preprocess_corpus(test_texts,  lowercase=True,
                                remove_stopwords=True, lemmatize=True)

print('Preprocessing: +stopwords +lemma +negation...')
train_neg = preprocess_corpus(train_texts, lowercase=True,
                               remove_stopwords=True, lemmatize=True,
                               handle_negation=True)
test_neg  = preprocess_corpus(test_texts,  lowercase=True,
                               remove_stopwords=True, lemmatize=True,
                               handle_negation=True)

print('Done.')

## 2. Experiment runner

In [ ]:
all_results = []

def run_experiment(name, model, vectorizer, train_corpus, test_corpus,
                   train_y, test_y, prep_label, task='2.3'):
    pipe = Pipeline([('vec', vectorizer), ('clf', model)])
    pipe.fit(train_corpus, train_y)
    preds = pipe.predict(test_corpus)
    metrics = evaluate_predictions(test_y, preds)
    print(f'{name} | {prep_label}: {metrics}')
    save_results(task, name, metrics, preprocessing=prep_label)
    all_results.append({'model': name, 'preprocessing': prep_label, **metrics})
    return pipe, metrics

## 3. Bag-of-Words experiments

In [ ]:
BOW = lambda: CountVectorizer(max_features=10_000, ngram_range=(1,1))

# LR + BoW
run_experiment('LR (BoW)',  LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
               BOW(), train_base, test_base, train_labels, test_labels, 'lowercase, BoW 10K')

run_experiment('LR (BoW)', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
               BOW(), train_full, test_full, train_labels, test_labels, 'lowercase+stopwords+lemma, BoW 10K')

# NB + BoW
run_experiment('NB (BoW)',  MultinomialNB(alpha=1.0),
               BOW(), train_full, test_full, train_labels, test_labels, 'lowercase+stopwords+lemma, BoW 10K')

# SVM + BoW
run_experiment('SVM (BoW)', LinearSVC(C=1.0, max_iter=2000),
               BOW(), train_full, test_full, train_labels, test_labels, 'lowercase+stopwords+lemma, BoW 10K')

## 4. TF-IDF experiments

In [ ]:
TFIDF_UNI  = lambda: TfidfVectorizer(max_features=10_000, ngram_range=(1,1))
TFIDF_BIGR = lambda: TfidfVectorizer(max_features=10_000, ngram_range=(1,2))

# LR + TF-IDF unigrams
run_experiment('LR (TF-IDF uni)', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
               TFIDF_UNI(), train_full, test_full, train_labels, test_labels,
               'lowercase+stopwords+lemma, TF-IDF 10K uni')

# LR + TF-IDF bigrams
run_experiment('LR (TF-IDF bi)', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
               TFIDF_BIGR(), train_full, test_full, train_labels, test_labels,
               'lowercase+stopwords+lemma, TF-IDF 10K bi')

# SVM + TF-IDF unigrams (best expected)
run_experiment('SVM (TF-IDF uni)', LinearSVC(C=1.0, max_iter=2000),
               TFIDF_UNI(), train_full, test_full, train_labels, test_labels,
               'lowercase+stopwords+lemma, TF-IDF 10K uni')

# SVM + TF-IDF bigrams + negation
run_experiment('SVM (TF-IDF bi + neg)', LinearSVC(C=1.0, max_iter=2000),
               TFIDF_BIGR(), train_neg, test_neg, train_labels, test_labels,
               'lowercase+stopwords+lemma+negation, TF-IDF 10K bi')

## 5. Hyperparameter tuning — best model

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('vec', TfidfVectorizer(max_features=10_000, ngram_range=(1,2))),
    ('clf', LinearSVC(max_iter=2000))
])
param_grid = {'clf__C': [0.1, 0.5, 1.0, 5.0, 10.0]}

gs = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
gs.fit(train_neg, train_labels)

print(f'Best C: {gs.best_params_} | CV accuracy: {gs.best_score_:.4f}')
best_preds = gs.predict(test_neg)
best_metrics = evaluate_predictions(test_labels, best_preds)
print('Best SVM test metrics:', best_metrics)
save_results('2.3', f'SVM (TF-IDF bi + neg, C={gs.best_params_["clf__C"]})',
             best_metrics,
             preprocessing='lowercase+stopwords+lemma+negation, TF-IDF 10K bi',
             notes=f'GridSearchCV 5-fold, best C={gs.best_params_["clf__C"]}')

## 6. Results summary

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(all_results).sort_values('accuracy', ascending=False)
display(df)

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df))
ax.bar(x, df['accuracy'], color='#4878D0', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['model']}\n{r['preprocessing'][:30]}" 
                    for _, r in df.iterrows()], rotation=45, ha='right', fontsize=7)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Task 2.3 — Classical ML: Accuracy by configuration')
plt.tight_layout()
plt.savefig('../results/fig_classical_ml.png', dpi=150, bbox_inches='tight')
plt.show()